In [ ]:
#!pip install bertopic

In [1]:
import os
import pandas as pd
from datetime import datetime
from openai import OpenAI
from bertopic import BERTopic
import numpy as np

In [2]:
# Folder where your CSV files are stored
data_folder = 'data/'

# Function to extract metadata from the file name
def extract_metadata(filename):
    parts = filename.split('_')
    action_item = int(parts[0])
    section = parts[1]
    date = datetime.strptime(parts[2].replace('.csv', ''), '%m%d%y').strftime('%m/%d/%Y')
    return action_item, section, date

# Process all CSV files in the data folder and merge them into one
def process_all_files(data_folder):
    all_data = []  # List to store all processed data
    short_lines_data = []  # List to store lines with <= 3 words
    total_lines_across_files = 0

    # Iterate through all files in the folder
    for filename in os.listdir(data_folder):
        if filename.endswith('.csv'):
            try:
                # Extract metadata from file name
                action_item, section, date = extract_metadata(filename)

                # Load the CSV file as plain text and treat each line as a chunk
                file_path = os.path.join(data_folder, filename)
                with open(file_path, 'r') as file:
                    lines = file.readlines()
                
                # Count the total number of lines before stripping
                total_lines_before_stripping = len(lines)
                total_lines_across_files += total_lines_before_stripping  # Add to the overall total
                print(f"{filename}: Total lines before stripping: {total_lines_before_stripping}")

                # Clean the lines
                lines = [line.strip() for line in lines if line.strip()]

                # Separate lines based on word count
                long_lines = [line for line in lines if len(line.split()) > 8]
                short_lines = [line for line in lines if len(line.split()) <= 8]

                # Create a DataFrame where each long line is a row
                if long_lines:
                    df = pd.DataFrame(long_lines, columns=['text'])
                    df['action_item'] = action_item
                    df['section'] = section
                    df['date'] = date
                    all_data.append(df)

                # Store short lines in a separate DataFrame
                if short_lines:
                    short_df = pd.DataFrame(short_lines, columns=['text'])
                    short_df['action_item'] = action_item
                    short_df['section'] = section
                    short_df['date'] = date
                    short_lines_data.append(short_df)

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Merge all the DataFrames for long lines into one
    merged_df = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

    # Merge all the DataFrames for short lines into one
    short_lines_df = pd.concat(short_lines_data, ignore_index=True) if short_lines_data else pd.DataFrame()

    # Print the final total number of lines across all files
    #print(f"\nTotal number of lines across all files: {total_lines_across_files}")

    # Return both DataFrames
    return merged_df, short_lines_df

# Run the function to process all files and return the DataFrames
long_lines_df_8, short_lines_df_8 = process_all_files(data_folder)

0_C2_112123.csv: Total lines before stripping: 27
0_C1_052124.csv: Total lines before stripping: 45
0_C2_041624.csv: Total lines before stripping: 29
1_B4_081523.csv: Total lines before stripping: 14
1_B4_072324.csv: Total lines before stripping: 65
1_B3_013024.csv: Total lines before stripping: 31
0_C2_051623.csv: Total lines before stripping: 26
1_B2_091923.csv: Total lines before stripping: 44
0_C2_031924.csv: Total lines before stripping: 28
1_B1_072324.csv: Total lines before stripping: 32
1_B6_013024.csv: Total lines before stripping: 16
0_C1_062023.csv: Total lines before stripping: 181
0_C1_041823.csv: Total lines before stripping: 35
0_C3_061824.csv: Total lines before stripping: 48
1_C2_072623.csv: Total lines before stripping: 39
0_B3_022024.csv: Total lines before stripping: 74
0_B1_032123.csv: Total lines before stripping: 31
1_B3_041823.csv: Total lines before stripping: 10
1_B1_061824.csv: Total lines before stripping: 7
1_B3_062023.csv: Total lines before stripping: 191

In [3]:
short_lines_df_8.tail(15)

,text,action_item,section,date
1249,Turnaround time couple of weeks?,0,C1,11/21/2023
1250,"Public accountability, I’m sure we’ll be respo...",0,C1,11/21/2023
1251,"So, no results yet.",0,C1,11/21/2023
1252,SMEs as well not just...,0,C1,11/21/2023
1253,"I can't, I have no aversion if there...",0,C1,11/21/2023
1254,We meter it at the well.,0,C1,11/21/2023
1255,We have no issue with adding the “shall.”,1,B1,09/19/2023
1256,does DOT do any kind of monitoring?,0,C1,05/16/2023
1257,that falls under the Department of Health.,0,C1,05/16/2023
1258,more money to hire USGS.,0,C1,05/16/2023


In [4]:
from itertools import chain 

In [8]:
long_lines = long_lines_df_8["text"].tolist()
split_lines = list(chain(*[x.split(". ") for x in long_lines]))
split_lines = [x for x in split_lines if len(x) > 5]
split_lines

['Informational Briefing by University of Hawai‘i Biota Survey in Hālawa Stream, O‘ahu',
 'Yinphan Tsang, University of Hawaiʻi, Associate Professor in Hydrology',
 'Ayron Strauch introduced Dr',
 'Tsang and her team from the University of Hawaiʻi',
 'She explained the effects of the Navy’s water discharge from the Hālawa Shaft into South Hālawa Stream.',
 'Thank you for your research',
 'Seems like the rising tide floated all the species or it benefited all the species',
 'Will over time the non-native species propose a habitat transforming risk because they outcompete or they are predators of the natives?',
 "That's definitely a very unfortunate situation but that's what we are seeing right now and so far",
 'You are probably already aware that we are experiencing a quite unusual drought situation this year',
 "I think that's also contributing to this unprecedented dry condition that might have increased this environmental stressor like hotter environment and allow those more toleran

In [ ]:
len(split_lines)

In [6]:
client = OpenAI()

In [ ]:
# # Function to get embeddings from OpenAI
# def get_embedding(text):
#     response = client.embeddings.create(model="text-embedding-3-small", input=text)
#     return response.data[0].embedding

# # Add embeddings to the long_lines_df
# def add_embeddings_to_df(df):
#     # Apply the get_embedding function to the 'text' column
#     df['embedding'] = df['text'].apply(get_embedding)
#     return df


In [9]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = stopwords.words('english')
print(stop_words)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

[nltk_data] Downloading package stopwords to /Users/mahdi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
split_lines

['Informational Briefing by University of Hawai‘i Biota Survey in Hālawa Stream, O‘ahu',
 ' Yinphan Tsang, University of Hawaiʻi, Associate Professor in Hydrology',
 ' Ayron Strauch introduced Dr',
 ' Tsang and her team from the University of Hawaiʻi',
 ' She explained the effects of the Navy’s water discharge from the Hālawa Shaft into South Hālawa Stream',
 'Thank you for your research',
 ' Seems like the rising tide floated all the species or it benefited all the species',
 ' Will over time the non-native species propose a habitat transforming risk because they outcompete or they are predators of the natives?',
 "That's definitely a very unfortunate situation but that's what we are seeing right now and so far",
 ' You are probably already aware that we are experiencing a quite unusual drought situation this year',
 " I think that's also contributing to this unprecedented dry condition that might have increased this environmental stressor like hotter environment and allow those more 

In [23]:
(split_lines)

16382

In [10]:
import torch
from sentence_transformers import SentenceTransformer

# load model with tokenizer
model = SentenceTransformer('nvidia/NV-Embed-v2', trust_remote_code=True)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
model.max_seq_length = 32768
model.tokenizer.padding_side="right"

def add_eos(input_examples):
  input_examples = [input_example + model.tokenizer.eos_token for input_example in input_examples]
  return input_examples

# get the embeddings
batch_size = 10
query_embeddings = model.encode(add_eos(split_lines), batch_size=batch_size, normalize_embeddings=True)



In [ ]:
from bertopic.representation import OpenAI
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words="english")
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = OpenAI(client, delay_in_seconds=10, model="gpt-4o-mini",chat=True)

# Assuming 'text' and 'embedding' columns are in the long_lines_df
texts = long_lines_df_8_with_embeddings['text'].tolist()  # Extract text
embeddings = np.array(long_lines_df_8_with_embeddings['embedding'].tolist())  # Convert embeddings to numpy array

# Create a BERTopic model
topic_model = BERTopic(
    nr_topics="auto",                 # Reduce the number of topics (int or None)
    top_n_words=15,                 # The number of words per topic
    min_topic_size=15,              # Minimum size of a topic cluster
    low_memory=False,               # Reduce memory usage at the cost of slower speed
    embedding_model=None,           # Model used to embed the text (you can use custom embeddings or models like BERT)
    vectorizer_model=vectorizer_model,        
    calculate_probabilities=False,  # Calculate the probability of topics per document
    seed_topic_list=None,           # Seed topics with certain words
    language="english",             # The language of the model
    verbose=False,                   # Set to True to get detailed output
    representation_model=representation_model
)

# Fit BERTopic on the texts and precomputed embeddings
topics, probabilities = topic_model.fit_transform(texts, embeddings)


In [ ]:
# Reduce outliers using the `embeddings` strategy
new_topics = topic_model.reduce_outliers(texts, topics, strategy="embeddings", embeddings=embeddings)

In [ ]:
# Add the topic labels to the DataFrame
long_lines_df_8_with_embeddings['topic'] = new_topics

# Display the DataFrame with topics
long_lines_df_8_with_embeddings[['text', 'topic']].head()


In [ ]:
# Visualize the topics
topic_model.visualize_topics()

In [ ]:
long_lines_df_8_with_embeddings.to_csv('long_lines_with_embeddings.csv', index=False)

In [ ]:
topic_model.get_topic(6)[0][0]


In [ ]:
# Ensure the 'date' column is in datetime format
long_lines_df_8_with_embeddings['date'] = pd.to_datetime(long_lines_df_8_with_embeddings['date'])

# Group by 'date' and 'topic', then count the number of occurrences of each topic per date
topic_counts_per_date = long_lines_df_8_with_embeddings.groupby(['date', 'topic']).size().reset_index(name='count')

# Display the result
print(topic_counts_per_date)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your data is already in a DataFrame called df
topic_counts_per_date['date'] = pd.to_datetime(topic_counts_per_date['date'])  # Ensure 'date' is a datetime object

# Get the unique dates from the DataFrame
topic_counts_per_date['normalized_count'] = topic_counts_per_date.groupby('date')['count'].transform(lambda x: x / x.sum())

unique_dates = topic_counts_per_date['date'].unique()

# Loop through each date and create a bar chart
for date in unique_dates:
    # Filter data for the current date
    date_df = topic_counts_per_date[topic_counts_per_date['date'] == date]
    
    # Create the bar plot for normalized counts
    plt.figure(figsize=(10, 6))
    sns.barplot(x='topic', y='normalized_count', data=date_df, palette='viridis')

    # Add titles and labels
    plt.title(f"Normalized Topic Distribution on {date.date()}", fontsize=16)
    plt.xlabel("Topic", fontsize=12)
    plt.ylabel("Normalized Count", fontsize=12)

    # Show the plot
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
pd.set_option('display.max_colwidth', None)

# Ensure the 'date' column is in datetime format
topic_counts_per_date['date'] = pd.to_datetime(topic_counts_per_date['date'])

# # Step 1: Group by 'date' and 'topic', then count the number of occurrences of each topic per date
# topic_counts_per_date = long_lines_df_8_with_embeddings.groupby(['date', 'topic']).size().reset_index(name='count')

# Step 2: Sort by 'date' and 'count' (descending order to get the most frequent topics first)
topic_counts_per_date = topic_counts_per_date.sort_values(by=['date', 'normalized_count'], ascending=[True, False])

# Step 3: Group by 'date' and select the top 2 topics for each date
top_2_topics_per_date = topic_counts_per_date.groupby('date').head(3)

# Display the result
#print(top_2_topics_per_date)

# Perform the merge, specifying the column names to merge on
merged_df = pd.merge(top_2_topics_per_date, topic_model.get_topic_info(), left_on='topic', right_on='Topic', how='left')

# Optionally, drop the 'Topic' column from df_with_titles after the merge if you don't need both 'topic' and 'Topic'
merged_df = merged_df.drop(columns=['Topic',"Representation","Representative_Docs","Count"])

# Display the merged DataFrame
merged_df

In [ ]:

grouped = merged_df.groupby('date')

# Loop through each group (each date) and print the top 3 topics separately
for date, group in grouped:
    print(f"Top 3 topics for {date}:")
    print(group[['topic', 'normalized_count']])
    print("\n" + "="*40 + "\n")

In [ ]:
topic_model.get_topic_freq()
# Perform the merge, specifying the column names to merge on
merged_df = pd.merge(topic_model.get_topic_freq(), topic_model.get_topic_info(), on='Topic', how='left')

# Optionally, drop the 'Topic' column from df_with_titles after the merge if you don't need both 'topic' and 'Topic'
merged_df = merged_df.drop(columns=["Name","Representative_Docs"])

# Display the merged DataFrame
merged_df

In [ ]:
topic_model.reduce_topics(texts, nr_topics=20)
topic_model.get_topic_freq()
# Perform the merge, specifying the column names to merge on
merged_df = pd.merge(topic_model.get_topic_freq(), topic_model.get_topic_info(), on='Topic', how='left')

# Optionally, drop the 'Topic' column from df_with_titles after the merge if you don't need both 'topic' and 'Topic'
merged_df = merged_df.drop(columns=["Name","Representative_Docs"])

# Display the merged DataFrame
merged_df